
# 02 · Rede

Constrói a rede de fluxo de internações entre os municípios do RN.

**Modelagem**
- **Nó:** município do RN (167)
- **Aresta:** residência → internação, apenas quando os municípios são diferentes
- **Direção:** dirigida (A envia pacientes para B)
- **Peso:** número de internações em 2024
- **Fora da rede:** residentes de outros estados (0,9%); internações no próprio município viram o atributo `autossuficiencia` do nó, não um laço

**Saídas:** `data/processed/rede_rn_2024.graphml` (abre no Gephi) e `data/processed/nos_rn_2024.csv`

In [2]:
from pathlib import Path

import networkx as nx
import pandas as pd

PROCESSED = Path("data/processed")
ANO = 2024
MIN_VOLUME = 30  # mínimo de internações fora do município para calcular dependência

df = pd.read_parquet(PROCESSED / f"rd_rn_{ANO}.parquet")
municipios = pd.read_csv(PROCESSED / "municipios_rn.csv", dtype={"cod6": str})

## 1. Filtro: só residentes do RN

In [3]:
fora_do_rn = df["NOME_RES"].isna()
print(f"Removidas (residentes de outros estados): {fora_do_rn.sum():,}")
df = df[~fora_do_rn]

local = df["MUNIC_RES"] == df["MUNIC_MOV"]
print(f"Internações no próprio município: {local.mean():.1%}")
print(f"Internações em outro município:   {(~local).mean():.1%}")

Removidas (residentes de outros estados): 1,977
Internações no próprio município: 46.0%
Internações em outro município:   54.0%


## 2. Atributos dos nós

- `internacoes_residentes`: total de internações de moradores do município
- `autossuficiencia`: fração dessas internações feita no próprio município

In [4]:
nos = municipios.set_index("cod6")
nos["internacoes_residentes"] = df.groupby("MUNIC_RES").size()
nos["internacoes_locais"] = df[local].groupby("MUNIC_RES").size()
nos = nos.fillna(0).astype({"internacoes_residentes": int, "internacoes_locais": int})
nos["autossuficiencia"] = nos["internacoes_locais"] / nos["internacoes_residentes"]

nos.sort_values("internacoes_residentes", ascending=False).head()

,nome,internacoes_residentes,internacoes_locais,autossuficiencia
cod6,,,,
240810,Natal,48335,46527,0.962594
240800,Mossoró,14438,12952,0.897077
240325,Parnamirim,11333,4620,0.407659
241200,São Gonçalo do Amarante,8495,4104,0.483108
240260,Ceará-Mirim,5971,2886,0.483336


## 3. Arestas: residência → internação

In [5]:
arestas = (df[~local]
           .groupby(["MUNIC_RES", "MUNIC_MOV"]).size()
           .rename("weight").reset_index())

print(f"Arestas: {len(arestas)}")
arestas.nlargest(10, "weight").assign(
    origem=lambda a: a["MUNIC_RES"].map(nos["nome"]),
    destino=lambda a: a["MUNIC_MOV"].map(nos["nome"]),
)

Arestas: 1709


,MUNIC_RES,MUNIC_MOV,weight,origem,destino
379,240325,240810,6485,Parnamirim,Natal
1355,241200,240810,3797,São Gonçalo do Amarante,Natal
787,240710,240810,3060,Macaíba,Natal
308,240260,240810,2730,Ceará-Mirim,Natal
421,240360,240810,2362,Extremoz,Natal
1374,241220,240810,2069,São José de Mipibu,Natal
16,240020,240800,1474,Assú,Mossoró
1626,241440,240810,1361,Touros,Natal
658,240580,240810,1357,João Câmara,Natal
909,240810,240325,1231,Natal,Parnamirim


## 4. Grafo

In [7]:
G = nx.DiGraph()
G.add_nodes_from(nos.index)
G.add_weighted_edges_from(arestas.itertuples(index=False, name=None))

print(G)
print(f"Densidade: {nx.density(G):.4f}")

DiGraph with 167 nodes and 1709 edges
Densidade: 0.0616


## 5. Grau de entrada e de saída

- **Grau:** número de municípios distintos
- **Força (grau ponderado):** número de internações

In [8]:
nos["grau_entrada"] = pd.Series(dict(G.in_degree()))
nos["grau_saida"] = pd.Series(dict(G.out_degree()))
nos["forca_entrada"] = pd.Series(dict(G.in_degree(weight="weight")))
nos["forca_saida"] = pd.Series(dict(G.out_degree(weight="weight")))

recebe = nos["grau_entrada"] > 0
envia = nos["grau_saida"] > 0
print(f"Recebem pacientes de fora: {recebe.sum()}")
print(f"Só enviam:                 {(envia & ~recebe).sum()}")
print(f"Isolados:                  {(~envia & ~recebe).sum()}")

nos.nlargest(10, "forca_entrada")[["nome", "grau_entrada", "forca_entrada", "autossuficiencia"]]

Recebem pacientes de fora: 57
Só enviam:                 110
Isolados:                  0


,nome,grau_entrada,forca_entrada,autossuficiencia
cod6,,,,
240810,Natal,166,61950,0.962594
240800,Mossoró,99,14891,0.897077
240940,Pau dos Ferros,66,5150,0.680332
240325,Parnamirim,154,4981,0.407659
241120,Santa Cruz,111,4083,0.546577
241220,São José de Mipibu,58,3899,0.195284
240310,Currais Novos,145,3382,0.544101
240200,Caicó,61,3185,0.728726
240060,Almino Afonso,128,2741,0.572505


## 6. Dependência de polo único

Entre as internações feitas **fora** do município, a fração que vai para o destino principal. Só para municípios com pelo menos `MIN_VOLUME` internações fora, para evitar frações calculadas sobre poucos casos.

In [9]:
principal = arestas.loc[arestas.groupby("MUNIC_RES")["weight"].idxmax()].set_index("MUNIC_RES")

nos["destino_principal"] = principal["MUNIC_MOV"].map(nos["nome"])
nos["dependencia"] = principal["weight"] / nos["forca_saida"]
nos.loc[nos["forca_saida"] < MIN_VOLUME, "dependencia"] = float("nan")

print(f"Municípios avaliados: {nos['dependencia'].notna().sum()} de {len(nos)}")
print(f"Dependência >= 70%:   {(nos['dependencia'] >= 0.7).sum()}")

nos["destino_principal"].value_counts().head()

Municípios avaliados: 167 de 167
Dependência >= 70%:   28


destino_principal
Natal             95
Mossoró           37
Pau dos Ferros    25
Caicó              9
Parnamirim         1
Name: count, dtype: int64

## 7. Salvar

In [10]:
for cod, attrs in nos.to_dict("index").items():
    G.nodes[cod].update({k: v for k, v in attrs.items() if pd.notna(v)})

nx.write_graphml(G, PROCESSED / f"rede_rn_{ANO}.graphml")
nos.to_csv(PROCESSED / f"nos_rn_{ANO}.csv")